# Text Classification Baseline (Keras)

Generated by `flashkeras.notebooks`.

Assumes a CSV with one text column and one label column.
Edit the **Parameters** cell below, then `Run All`.

In [ ]:
csv_path = "REPLACE_ME.csv"
text_column = "text"
label_column = "label"
max_tokens = 20000
sequence_length = 250
embedding_dim = 128
batch_size = 32
validation_split = 0.2
epochs = 10
seed = 42


## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

# Uncomment if you want to use flashkeras helpers directly in this notebook:
# import flashkeras

print("TensorFlow:", tf.__version__)


## 2. Load data

In [ ]:
df = pd.read_csv(csv_path)
df = df[[text_column, label_column]].dropna()
print(f"Rows: {len(df)}")
df.head()


## 3. Label distribution

In [ ]:
df[label_column].value_counts().plot(kind="bar", title="Label distribution")
plt.show()


## 4. Encode labels

In [ ]:
labels = df[label_column].astype("category")
label_names = labels.cat.categories.tolist()
y = labels.cat.codes.to_numpy()
num_classes = len(label_names)
print(f"Classes ({num_classes}): {label_names}")


## 5. Train / validation split

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    df[text_column].to_numpy(),
    y,
    test_size=validation_split,
    random_state=seed,
    stratify=y,
)

train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train)).batch(batch_size)
val_ds = tf.data.Dataset.from_tensor_slices((X_val, y_val)).batch(batch_size)


## 6. Text vectorization

In [ ]:
vectorize_layer = layers.TextVectorization(
    max_tokens=max_tokens,
    output_mode="int",
    output_sequence_length=sequence_length,
)

vectorize_layer.adapt(train_ds.map(lambda text, label: text))


## 7. Build baseline model

In [ ]:
model = keras.Sequential([
    layers.Input(shape=(1,), dtype=tf.string),
    vectorize_layer,
    layers.Embedding(max_tokens, embedding_dim, mask_zero=True),
    layers.GlobalAveragePooling1D(),
    layers.Dropout(0.3),
    layers.Dense(64, activation="relu"),
    layers.Dense(num_classes, activation="softmax"),
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

model.summary()


## 8. Train

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=epochs,
    callbacks=callbacks,
)


## 9. Training curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history["accuracy"], label="train")
axes[0].plot(history.history["val_accuracy"], label="val")
axes[0].set_title("Accuracy")
axes[0].legend()

axes[1].plot(history.history["loss"], label="train")
axes[1].plot(history.history["val_loss"], label="val")
axes[1].set_title("Loss")
axes[1].legend()

plt.tight_layout()
plt.show()


## 10. Evaluate

In [ ]:
val_loss, val_acc = model.evaluate(val_ds)
print(f"Validation accuracy: {val_acc:.4f}")


## Next steps

- Try pretrained embeddings or a small transformer instead of average pooling
- Tune `max_tokens` and `sequence_length` based on your text length distribution
- Add class weights if `label_column` is imbalanced
- Save the model with `model.save("model.keras")`